# E052 — Robustness augmentation: codec aug (audio) + Cutout (image)

E051 identified two vulnerabilities:
- Audio codec (8kHz BW): 13.33% EER — LPCC loses upper formants
- Image occlusion: 11.06% EER — PCA has no spatial locality

This notebook tests targeted augmentations for each.

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../src').resolve()))

import copy
import io

import librosa
import numpy as np
from PIL import Image
from scipy.ndimage import rotate as nd_rotate
from scipy.special import logsumexp
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

SEED = 67
DATA = Path('../data').resolve()
manifest = load_manifest(DATA)
print(f'{len(manifest)} samples loaded')

222 samples loaded


## Image helpers

In [2]:
N_PCA, C_LR = 50, 1.0

def _find_png(stem, data_dir):
    for sf in ('target_train','target_dev','non_target_train','non_target_dev'):
        p = data_dir / sf / (stem + '.png')
        if p.exists(): return p
    raise FileNotFoundError(stem)

def _load_image(path):
    img = Image.open(path).convert('RGB')
    if img.size != (80, 80):
        img = img.resize((80, 80), Image.BILINEAR)
    return np.array(img, dtype=np.float32).mean(axis=2).flatten()

# --- train augmentations ---
def _aug_flip(x): return x.reshape(80,80)[:,::-1].flatten()
def _aug_brightness(x, rng): return np.clip(x * rng.uniform(0.7,1.3), 0, 255)
def _aug_noise(x, rng): return np.clip(x + rng.normal(0,15,x.shape), 0, 255)
def _aug_rotate(x, angle):
    return nd_rotate(x.reshape(80,80), angle, reshape=False,
                     order=1, mode='constant', cval=0).flatten()

def _aug_cutout(x, rng, size=20):
    """E052: mask a random 20x20 patch to 0. Same patch size as occlude stress."""
    img = x.reshape(80, 80).copy()
    r = rng.integers(0, 80 - size)
    c = rng.integers(0, 80 - size)
    img[r:r+size, c:c+size] = 0
    return img.flatten()

def _find_adv_rotation(x, scaler, pca, clf, max_angle=10.0, n_steps=5):
    best_angle, worst_abs = 0.0, np.inf
    for angle in np.linspace(-max_angle, max_angle, n_steps):
        if abs(angle) < 0.1: continue
        logit = clf.decision_function(
            pca.transform(scaler.transform(_aug_rotate(x, angle).reshape(1,-1)))
        )[0]
        if abs(logit) < worst_abs:
            worst_abs, best_angle = abs(logit), angle
    return best_angle

# --- val-time stresses ---
def _stress_occlude(x, size=20, rng=None):
    if rng is None: rng = np.random.default_rng(42)
    img = x.reshape(80,80).copy()
    r = rng.integers(0, 80-size)
    c = rng.integers(0, 80-size)
    img[r:r+size, c:c+size] = 0
    return img.flatten()

print('Image helpers ready')

Image helpers ready


## Image models: E033 baseline vs E033+Cutout

In [3]:
def _build_basic_aug(df, data_dir, seed):
    rng = np.random.default_rng(seed)
    X, y = [], []
    for _, row in df.iterrows():
        orig = _load_image(_find_png(row['stem'], data_dir))
        for v in [orig, _aug_flip(orig), _aug_brightness(orig,rng), _aug_noise(orig,rng)]:
            X.append(v); y.append(row['label'])
    return np.stack(X), np.array(y), rng

def _train_e033(df, data_dir, seed):
    """E033 baseline: +All aug + adversarial rotation."""
    X_basic, y_basic, _ = _build_basic_aug(df, data_dir, seed)
    scaler = StandardScaler()
    pca = PCA(n_components=N_PCA, random_state=SEED)
    clf = LogisticRegression(C=C_LR, max_iter=1000, random_state=SEED)
    clf.fit(pca.fit_transform(scaler.fit_transform(X_basic)), y_basic)

    X_adv, y_adv = [], []
    for _, row in df.iterrows():
        orig = _load_image(_find_png(row['stem'], data_dir))
        angle = _find_adv_rotation(orig, scaler, pca, clf)
        X_adv.append(_aug_rotate(orig, angle)); y_adv.append(row['label'])

    X_all = np.vstack([X_basic, np.stack(X_adv)])
    y_all = np.concatenate([y_basic, np.array(y_adv)])
    scaler = StandardScaler()
    pca = PCA(n_components=N_PCA, random_state=SEED)
    clf = LogisticRegression(C=C_LR, max_iter=1000, random_state=SEED)
    clf.fit(pca.fit_transform(scaler.fit_transform(X_all)), y_all)
    return scaler, pca, clf

def _train_e033_cutout(df, data_dir, seed):
    """E052 image: E033 + Cutout aug (random 20x20 patch per sample)."""
    rng = np.random.default_rng(seed)
    X_basic, y_basic = [], []
    for _, row in df.iterrows():
        orig = _load_image(_find_png(row['stem'], data_dir))
        for v in [orig, _aug_flip(orig), _aug_brightness(orig,rng), _aug_noise(orig,rng),
                  _aug_cutout(orig, rng)]:
            X_basic.append(v); y_basic.append(row['label'])
    X_basic = np.stack(X_basic); y_basic = np.array(y_basic)

    scaler = StandardScaler()
    pca = PCA(n_components=N_PCA, random_state=SEED)
    clf = LogisticRegression(C=C_LR, max_iter=1000, random_state=SEED)
    clf.fit(pca.fit_transform(scaler.fit_transform(X_basic)), y_basic)

    X_adv, y_adv = [], []
    for _, row in df.iterrows():
        orig = _load_image(_find_png(row['stem'], data_dir))
        angle = _find_adv_rotation(orig, scaler, pca, clf)
        X_adv.append(_aug_rotate(orig, angle)); y_adv.append(row['label'])

    X_all = np.vstack([X_basic, np.stack(X_adv)])
    y_all = np.concatenate([y_basic, np.array(y_adv)])
    scaler = StandardScaler()
    pca = PCA(n_components=N_PCA, random_state=SEED)
    clf = LogisticRegression(C=C_LR, max_iter=1000, random_state=SEED)
    clf.fit(pca.fit_transform(scaler.fit_transform(X_all)), y_all)
    return scaler, pca, clf

def _score_image(x, scaler, pca, clf):
    return float(clf.decision_function(pca.transform(scaler.transform(x.reshape(1,-1))))[0])

print('Image model functions ready')

Image model functions ready


## Run image experiment (clean + occlude)

In [4]:
IMG_MODELS = {'E033 (baseline)': _train_e033, 'E033+Cutout': _train_e033_cutout}
IMG_CONDS  = ['clean', 'occlude']

img_results = {name: {c: [] for c in IMG_CONDS} for name in IMG_MODELS}
img_dcf     = {name: {c: [] for c in IMG_CONDS} for name in IMG_MODELS}

for model_name, train_fn in IMG_MODELS.items():
    print(f'\n=== {model_name} ===')
    for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
        print(f'  fold {fold_id}...', end=' ', flush=True)
        scaler, pca, clf = train_fn(manifest.loc[train_idx], DATA, SEED + fold_id)
        val_df = manifest.loc[val_idx]

        for cond in IMG_CONDS:
            scores, labels = [], []
            occlude_rng = np.random.default_rng(99)
            for _, row in val_df.iterrows():
                x = _load_image(_find_png(row['stem'], DATA))
                x_s = _stress_occlude(x, rng=occlude_rng) if cond == 'occlude' else x
                scores.append(_score_image(x_s, scaler, pca, clf))
                labels.append(row['label'])
            scores = np.array(scores); labels = np.array(labels)
            eer, _  = compute_eer(scores[labels==1], scores[labels==0])
            dcf, _  = compute_min_dcf(scores[labels==1], scores[labels==0])
            img_results[model_name][cond].append(eer * 100)
            img_dcf[model_name][cond].append(dcf)
        print('done')

print('\nImage experiment complete')


=== E033 (baseline) ===
  fold 0... 

done
  fold 1... 

done
  fold 2... 

done

=== E033+Cutout ===
  fold 0... 

done
  fold 1... 

done
  fold 2... 

done

Image experiment complete


In [5]:
print('\n=== IMAGE RESULTS (E033 baseline vs E033+Cutout) ===\n')
print(f'{"Config":<22} {"Clean EER":>12} {"Occlude EER":>14} {"Delta occlude":>15} {"Clean DCF":>11} {"Occlude DCF":>12}')
print('-' * 92)

for model_name in IMG_MODELS:
    clean_eer  = np.array(img_results[model_name]['clean'])
    occ_eer    = np.array(img_results[model_name]['occlude'])
    clean_dcf  = np.array(img_dcf[model_name]['clean'])
    occ_dcf    = np.array(img_dcf[model_name]['occlude'])
    delta = np.mean(occ_eer) - np.mean(clean_eer)
    sign = '+' if delta >= 0 else ''
    print(f'{model_name:<22}'
          f' {np.mean(clean_eer):>6.2f}±{np.std(clean_eer):.2f}%'
          f' {np.mean(occ_eer):>8.2f}±{np.std(occ_eer):.2f}%'
          f' {sign}{delta:>+8.2f}pp'
          f' {np.mean(clean_dcf):>10.4f}'
          f' {np.mean(occ_dcf):>11.4f}')

print('\nPer-fold breakdown:')
for model_name in IMG_MODELS:
    print(f'  {model_name}:')
    for cond in IMG_CONDS:
        folds = img_results[model_name][cond]
        print(f'    {cond:>8}: ' + '  '.join(f'F{i}={v:.2f}%' for i,v in enumerate(folds)))


=== IMAGE RESULTS (E033 baseline vs E033+Cutout) ===

Config                    Clean EER    Occlude EER   Delta occlude   Clean DCF  Occlude DCF
--------------------------------------------------------------------------------------------
E033 (baseline)          0.51±0.36%    10.32±6.60% +   +9.81pp     0.0102      0.1565
E033+Cutout              1.71±1.24%     6.81±3.94% +   +5.09pp     0.0343      0.1028

Per-fold breakdown:
  E033 (baseline):
       clean: F0=0.69%  F1=0.83%  F2=0.00%
     occlude: F0=8.47%  F1=19.17%  F2=3.33%
  E033+Cutout:
       clean: F0=3.47%  F1=0.83%  F2=0.83%
     occlude: F0=11.25%  F1=1.67%  F2=7.50%


## Audio helpers

In [6]:
UBM_COMPONENTS, MAP_R = 32, 16.0

def _find_wav(stem, data_dir):
    for sf in ('target_train','target_dev','non_target_train','non_target_dev'):
        p = data_dir / sf / (stem + '.wav')
        if p.exists(): return p
    raise FileNotFoundError(stem)

def _extract_lpcc(y, sr, order=12, n_cep=13, hop_length=160, win_length=400):
    frames = librosa.util.frame(y, frame_length=win_length, hop_length=hop_length)
    cep_frames = []
    for frame in frames.T:
        frame = frame * np.hanning(len(frame))
        try:
            a = librosa.lpc(frame.astype(np.float64), order=order)
            log_H = -np.log(np.abs(np.fft.rfft(a, n=512)) + 1e-10)
            cep = np.real(np.fft.irfft(log_H))[:n_cep]
        except Exception:
            cep = np.zeros(n_cep)
        cep_frames.append(cep)
    feat = np.array(cep_frames, dtype=np.float32)
    delta  = librosa.feature.delta(feat.T).T
    delta2 = librosa.feature.delta(feat.T, order=2).T
    feat = np.hstack([feat, delta, delta2])
    feat -= feat.mean(axis=0)
    return feat

def _train_ubm(X):
    return GaussianMixture(n_components=UBM_COMPONENTS,
                           covariance_type='tied', max_iter=200,
                           random_state=SEED).fit(X)

def _map_adapt(ubm, X_target):
    log_resp = ubm._estimate_log_prob(X_target) + np.log(ubm.weights_)
    log_resp -= logsumexp(log_resp, axis=1, keepdims=True)
    resp = np.exp(log_resp)
    n_k = resp.sum(axis=0)
    mu_hat = (resp.T @ X_target) / (n_k[:,None] + 1e-10)
    alpha = n_k / (n_k + MAP_R)
    adapted = copy.deepcopy(ubm)
    adapted.means_ = alpha[:,None]*mu_hat + (1-alpha[:,None])*ubm.means_
    return adapted

def _aug_codec(y, sr):
    """E052: simulate codec bandwidth loss by downsampling to 8kHz and back."""
    y_down = librosa.resample(y, orig_sr=sr, target_sr=8000)
    return librosa.resample(y_down, orig_sr=8000, target_sr=sr)

def _score_tta(y_orig, sr, adapted, ubm):
    """Speed TTA: average LLR over 0.9x/1.0x/1.1x (E042)."""
    views = [y_orig,
             librosa.effects.time_stretch(y_orig, rate=0.9),
             librosa.effects.time_stretch(y_orig, rate=1.1)]
    scores = []
    for v in views:
        f = _extract_lpcc(v, sr)
        scores.append(float((adapted.score_samples(f) - ubm.score_samples(f)).mean()))
    return float(np.mean(scores))

# codec stress for val
def _astress_codec(y, sr):
    y_down = librosa.resample(y, orig_sr=sr, target_sr=8000)
    return librosa.resample(y_down, orig_sr=8000, target_sr=sr)

print('Audio helpers ready')

Audio helpers ready


## Audio models: E042 baseline vs E042+codec_aug

In [7]:
def _build_frames(df, data_dir, seed, extra_codec=False):
    rng = np.random.default_rng(seed)
    X_list, y_list = [], []
    for _, row in df.iterrows():
        y_wav, sr = librosa.load(_find_wav(row['stem'], data_dir), sr=None, mono=True)
        wavs = [y_wav,
                librosa.effects.pitch_shift(y_wav, sr=sr,
                    n_steps=float(rng.choice([-2,-1,1,2])))]
        if extra_codec:
            wavs.append(_aug_codec(y_wav, sr))
        for y_aug in wavs:
            f = _extract_lpcc(y_aug, sr)
            X_list.append(f)
            y_list.extend([row['label']] * len(f))
    return np.vstack(X_list), np.array(y_list)

def _train_e042(df, data_dir, seed):
    """E042 baseline: LPCC + tied cov + pitch aug."""
    X, y = _build_frames(df, data_dir, seed, extra_codec=False)
    ubm = _train_ubm(X[y==0])
    return ubm, _map_adapt(ubm, X[y==1])

def _train_e042_codec(df, data_dir, seed):
    """E052 audio: E042 + codec aug (8kHz BW copy of each training wav)."""
    X, y = _build_frames(df, data_dir, seed, extra_codec=True)
    ubm = _train_ubm(X[y==0])
    return ubm, _map_adapt(ubm, X[y==1])

print('Audio model functions ready')

Audio model functions ready


## Run audio experiment (clean + codec)

In [8]:
AUD_MODELS = {'E042 (baseline)': _train_e042, 'E042+codec_aug': _train_e042_codec}
AUD_CONDS  = ['clean', 'codec']

aud_results = {name: {c: [] for c in AUD_CONDS} for name in AUD_MODELS}
aud_dcf     = {name: {c: [] for c in AUD_CONDS} for name in AUD_MODELS}

for model_name, train_fn in AUD_MODELS.items():
    print(f'\n=== {model_name} ===')
    for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
        print(f'  fold {fold_id}...', end=' ', flush=True)
        ubm, adapted = train_fn(manifest.loc[train_idx], DATA, SEED + fold_id)
        val_df = manifest.loc[val_idx]

        for cond in AUD_CONDS:
            scores, labels = [], []
            for _, row in val_df.iterrows():
                y_orig, sr = librosa.load(_find_wav(row['stem'], DATA), sr=None, mono=True)
                y_s = _astress_codec(y_orig, sr) if cond == 'codec' else y_orig
                scores.append(_score_tta(y_s, sr, adapted, ubm))
                labels.append(row['label'])
            scores = np.array(scores); labels = np.array(labels)
            eer, _  = compute_eer(scores[labels==1], scores[labels==0])
            dcf, _  = compute_min_dcf(scores[labels==1], scores[labels==0])
            aud_results[model_name][cond].append(eer * 100)
            aud_dcf[model_name][cond].append(dcf)
        print('done')

print('\nAudio experiment complete')


=== E042 (baseline) ===
  fold 0... 

done
  fold 1... 

done
  fold 2... 

done

=== E042+codec_aug ===
  fold 0... 

done
  fold 1... 

done
  fold 2... 

done

Audio experiment complete


In [9]:
print('\n=== AUDIO RESULTS (E042 baseline vs E042+codec_aug) ===\n')
print(f'{"Config":<22} {"Clean EER":>12} {"Codec EER":>13} {"Delta codec":>13} {"Clean DCF":>11} {"Codec DCF":>11}')
print('-' * 87)

for model_name in AUD_MODELS:
    clean_eer = np.array(aud_results[model_name]['clean'])
    codec_eer = np.array(aud_results[model_name]['codec'])
    clean_dcf = np.array(aud_dcf[model_name]['clean'])
    codec_dcf = np.array(aud_dcf[model_name]['codec'])
    delta = np.mean(codec_eer) - np.mean(clean_eer)
    sign = '+' if delta >= 0 else ''
    print(f'{model_name:<22}'
          f' {np.mean(clean_eer):>6.2f}±{np.std(clean_eer):.2f}%'
          f' {np.mean(codec_eer):>7.2f}±{np.std(codec_eer):.2f}%'
          f' {sign}{delta:>+8.2f}pp'
          f' {np.mean(clean_dcf):>10.4f}'
          f' {np.mean(codec_dcf):>10.4f}')

print('\nPer-fold breakdown:')
for model_name in AUD_MODELS:
    print(f'  {model_name}:')
    for cond in AUD_CONDS:
        folds = aud_results[model_name][cond]
        print(f'    {cond:>6}: ' + '  '.join(f'F{i}={v:.2f}%' for i,v in enumerate(folds)))


=== AUDIO RESULTS (E042 baseline vs E042+codec_aug) ===

Config                    Clean EER     Codec EER   Delta codec   Clean DCF   Codec DCF
---------------------------------------------------------------------------------------
E042 (baseline)          0.46±0.65%   13.33±3.79% +  +12.87pp     0.0093     0.1694
E042+codec_aug           0.46±0.65%    3.33±4.14% +   +2.87pp     0.0093     0.0333

Per-fold breakdown:
  E042 (baseline):
     clean: F0=1.39%  F1=0.00%  F2=0.00%
     codec: F0=18.33%  F1=9.17%  F2=12.50%
  E042+codec_aug:
     clean: F0=1.39%  F1=0.00%  F2=0.00%
     codec: F0=9.17%  F1=0.83%  F2=0.00%


## Summary

In [10]:
print('=== E052 SUMMARY ===')
print()

# Image
e033_clean  = np.mean(img_results['E033 (baseline)']['clean'])
e033_occ    = np.mean(img_results['E033 (baseline)']['occlude'])
cut_clean   = np.mean(img_results['E033+Cutout']['clean'])
cut_occ     = np.mean(img_results['E033+Cutout']['occlude'])

print('Image:')
print(f'  E033 baseline: clean={e033_clean:.2f}%, occlude={e033_occ:.2f}%')
print(f'  E033+Cutout:   clean={cut_clean:.2f}%, occlude={cut_occ:.2f}%')
img_verdict = 'ADOPT' if cut_clean <= e033_clean + 0.1 and cut_occ < e033_occ else 'REJECT'
print(f'  Verdict: {img_verdict} (rule: clean must not regress >0.1pp AND occlude must improve)')

print()

# Audio
e042_clean  = np.mean(aud_results['E042 (baseline)']['clean'])
e042_codec  = np.mean(aud_results['E042 (baseline)']['codec'])
caud_clean  = np.mean(aud_results['E042+codec_aug']['clean'])
caud_codec  = np.mean(aud_results['E042+codec_aug']['codec'])

print('Audio:')
print(f'  E042 baseline: clean={e042_clean:.2f}%, codec={e042_codec:.2f}%')
print(f'  E042+codec:    clean={caud_clean:.2f}%, codec={caud_codec:.2f}%')
aud_verdict = 'ADOPT' if caud_clean <= e042_clean + 0.1 and caud_codec < e042_codec else 'REJECT'
print(f'  Verdict: {aud_verdict} (rule: clean must not regress >0.1pp AND codec must improve)')

=== E052 SUMMARY ===

Image:
  E033 baseline: clean=0.51%, occlude=10.32%
  E033+Cutout:   clean=1.71%, occlude=6.81%
  Verdict: REJECT (rule: clean must not regress >0.1pp AND occlude must improve)

Audio:
  E042 baseline: clean=0.46%, codec=13.33%
  E042+codec:    clean=0.46%, codec=3.33%
  Verdict: ADOPT (rule: clean must not regress >0.1pp AND codec must improve)
